# NAS6 데이터 처리 — 강수 & 도로 위계

## 데이터 개요

| 데이터 | 파일 | 설명 |
|--------|------|------|
| 강수 | `days from the last rain_by grid.parquet` | 격자별 마지막 비 이후 일수 (2023-10-01 ~ 2025-10-31) |
| 도로 위계 | `서울 도로 위계 및 GVI.gpkg` | OSM 기반 도로 분류 + GVI (11,076개 구간) |

## 처리 결과 (RoadExtension_V2에 통합)
- `RoadExtension_V2/step_rain.py` → `checkpoints/rain_cache.csv`
- `RoadExtension_V2/step_road_hier.py` → `checkpoints/road_hier_grid.csv`

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import sqlite3

rain_path  = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/days from the last rain_by grid.parquet"
road_hier  = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/서울 도로 위계 및 GVI.gpkg"

## 1. 강수 데이터 탐색

In [ ]:
# 강수 파케이 구조 확인
pf = pq.ParquetFile(rain_path)
print("Schema:", pf.schema_arrow)
print("총 rows:", pf.metadata.num_rows)

sample = pf.read(columns=['CELL_ID','조사년월일','days from the last rain','일일강수량']).to_pandas()
sample['date'] = pd.to_datetime(sample['조사년월일'].astype(str))
print("\n날짜 범위:", sample['date'].min(), "~", sample['date'].max())
print("격자 수:", sample['CELL_ID'].nunique())
print("\n컬럼별 통계:")
print(sample[['days from the last rain','일일강수량']].describe())

In [ ]:
# 강수 데이터와 도로 PM 상관관계
import openpyxl

dfs = []
for yr in ['23','24','25']:
    wb = openpyxl.load_workbook(
        f'/home/data/youngwoong/ST-GNN_Dataset/0-2. 도로 미세먼지/서울 비재산먼지_{yr}년도.xlsx',
        data_only=True)
    rows = list(wb.active.iter_rows(values_only=True))
    wb.close()
    dfs.append(pd.DataFrame(rows[1:], columns=rows[0]))
road_df = pd.concat(dfs, ignore_index=True)
road_df['재비산먼지'] = pd.to_numeric(road_df['재비산먼지 평균농도(㎍/㎥)'], errors='coerce')
road_df['date'] = pd.to_datetime(road_df['측정일자']).dt.normalize()
road_df = road_df.dropna(subset=['재비산먼지'])

road_daily = road_df.groupby('date')['재비산먼지'].mean().reset_index()
rain_daily = sample.groupby('date')['days from the last rain'].mean().reset_index()
merged = road_daily.merge(rain_daily, on='date', how='inner')

corr = merged[['재비산먼지','days from the last rain']].corr().iloc[0,1]
print(f"상관계수 (일별 서울 평균): r={corr:.3f}")
print("→ 강수 이후 일수가 증가할수록 도로 PM 증가 (비 이후 도로 표면 건조 → 먼지 재비산 증가)")

# 구간별 평균
merged['rain_bin'] = pd.cut(merged['days from the last rain'],
    bins=[0,1,3,7,15,100], labels=['0-1일','1-3일','3-7일','7-15일','>15일'])
print("\n마지막 비 이후 일수별 평균 도로 PM:")
print(merged.groupby('rain_bin')['재비산먼지'].agg(['mean','count']))

## 2. 도로 위계 데이터 탐색

In [ ]:
# 도로 위계 구조
conn = sqlite3.connect(road_hier)
cursor = conn.cursor()
cursor.execute('PRAGMA table_info("서울 도로 위계 및 GVI")')
cols = cursor.fetchall()
print("컬럼:")
for c in cols:
    print(f"  {c[1]} ({c[2]})")

col_names = [c[1] for c in cols if c[1] != 'geom']
cursor.execute('SELECT {} FROM "서울 도로 위계 및 GVI" LIMIT 5'.format(
    ', '.join(f'"{c}"' for c in col_names)))
rows = cursor.fetchall()
df_road = pd.DataFrame(rows, columns=col_names)
print("\n샘플:")
print(df_road[['highway_type','highway_label','lanes_num','GVI','length']].to_string())

cursor.execute('SELECT COUNT(*) FROM "서울 도로 위계 및 GVI"')
print(f"\n총 도로 구간: {cursor.fetchone()[0]:,}")
conn.close()

In [ ]:
# 도로 위계별 분포
conn = sqlite3.connect(road_hier)
cursor = conn.cursor()
cursor.execute('SELECT "highway_label", COUNT(*), AVG("lanes_num"), AVG("GVI") FROM "서울 도로 위계 및 GVI" GROUP BY "highway_label"')
rows = cursor.fetchall()
conn.close()

df_summary = pd.DataFrame(rows, columns=['highway_label','count','avg_lanes','avg_gvi'])
print("도로 위계별 통계:")
print(df_summary.to_string())

## 3. 전처리 결과 확인

실제 처리는 RoadExtension_V2 스크립트에서 수행됨:
- `python3 RoadExtension_V2/step_rain.py`
- `python3 RoadExtension_V2/step_road_hier.py`

In [ ]:
# 처리된 캐시 파일 확인
rain_cache = pd.read_csv('/workspace/ST-GNN Modeling/RoadExtension_V2/checkpoints/rain_cache.csv', parse_dates=['date'])
hier_cache = pd.read_csv('/workspace/ST-GNN Modeling/RoadExtension_V2/checkpoints/road_hier_grid.csv')

print("=== 강수 캐시 ===")
print(f"Shape: {rain_cache.shape}")
print(f"기간: {rain_cache['date'].min().date()} ~ {rain_cache['date'].max().date()}")
print(rain_cache['days_from_rain'].describe())

print("\n=== 도로 위계 격자 캐시 ===")
print(f"Shape: {hier_cache.shape}")
print(hier_cache[['highway_rank','max_lanes','mean_gvi']].describe())

## 4. RoadExtension V2 Ablation 결과

| 실험 | 피처 | Test MAE | Test R² | 비고 |
|------|------|---------|---------|------|
| D_ambient | weather+LUR+ambient_pm | **7.35** | **0.3144** | **현재 최고** |
| G_rain | D_ambient + 강수 | 7.43 | 0.2979 | 효과 없음 |
| H_hier | D_ambient + 도로위계 | 7.31 | 0.3185 | 미미한 개선 |
| I_all_v2 | 전부 | 7.53 | 0.2734 | 과적합 |

### 강수 데이터가 효과 없는 이유
1. 학습셋 커버리지 62.3% vs 테스트셋 83.3% 불균형
2. 격자 수준에서 신호 약함 (r=0.236은 일별 서울 평균 기준)
3. 기온+습도+season이 이미 강수 조건 간접 학습

### 도로 위계가 효과 없는 이유
1. 격자 커버리지 26.2%만 매칭 (중점 기반 단순 매칭의 한계)
2. highway_rank 중요도 = 3 (거의 0)